# RG3 가스 모델 튜닝 및 CN7 전이
RG3 내부 검증으로만 설정 선택. CN7 라벨은 결과 평가에만 사용.

In [1]:
from pathlib import Path
import json,subprocess
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/24_rg3_to_cn7_zero_shot_transfer.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('# res_features uses')[0])
# This setup creates source RG3 and target CN7; no target labels enter fitting.
source=np.arange(source_n); target_ids=np.arange(source_n,len(d))
configs=[]
for representation in ['std4','relative_std4']:
    for depth,leaf,weight in [(8,3,'balanced_subsample'),(4,3,'balanced_subsample'),(8,8,'balanced_subsample'),(4,8,'balanced_subsample'),(8,3,None),(8,8,None)]:
        configs.append(dict(representation=representation,depth=depth,leaf=leaf,weight=weight))
X={}
X['std4']=d[stdcols].to_numpy(float)
denom=np.column_stack([(d[c.split('__')[0]]-d[c.replace('past10_std','dev10')]).abs().clip(lower=1).to_numpy(float) for c in stdcols])
X['relative_std4']=X['std4']/denom
predcache={}
def predict(c,ref,q):
    key=(c,tuple(ref),tuple(q))
    if key in predcache:return predcache[key]
    assert set(groups.iloc[ref]).isdisjoint(set(groups.iloc[q]))
    if len(np.unique(y[ref]))<2:
        z=np.full(len(q),float(y[ref][0]) if len(ref) else 0.)
    else:
        conf=configs[c]
        model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=conf['depth'],min_samples_leaf=conf['leaf'],class_weight=conf['weight'],random_state=42,n_jobs=-1))
        z=model.fit(X[conf['representation']][ref],y[ref]).predict_proba(X[conf['representation']][q])[:,1]
    predcache[key]=z
    return z
def oof(c,ref):
    z=np.full(len(ref),np.nan)
    for block in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(block).to_numpy()
        tr=ref[~mask];q=ref[mask]
        if len(tr):z[mask]=predict(c,tr,q)
    return z
def select(ref):
    vals=[];scores=[]
    for c in range(len(configs)):
        z=oof(c,ref); ok=np.isfinite(z)
        vals.append(average_precision_score(y[ref][ok],z[ok]) if ok.any() and y[ref][ok].sum() else -1)
        scores.append(z)
    best=int(np.argmax(vals))
    return best,scores[best],vals
rows=[];predrows=[];audit=[]
def evaluate(scheme,q,score,cal,cal_y,chosen,block):
    for budget in [.01,.03,.05]:
        normal=cal[(cal_y==0)&np.isfinite(cal)]
        cut=float(np.quantile(normal,1-budget,method='higher')) if len(normal) else 1.
        p=score>cut
        for i,k in enumerate(q):predrows.append(dict(scheme=scheme,block=block,config=chosen,budget=budget,TimeStamp=d.TimeStamp.iloc[k],actual=int(y[k]),score=float(score[i]),threshold=cut,prediction=int(p[i])))
for scheme in ['RG3_nested_leave_block_out','RG3_nested_past_only']:
    for j,block in enumerate(blocks):
        q=source[groups.iloc[source].eq(block)]
        ref=source[~groups.iloc[source].eq(block)] if scheme.endswith('leave_block_out') else source[groups.iloc[source].isin(blocks[:j])]
        if len(np.unique(y[ref]))<2:continue
        chosen,cal,vals=select(ref)
        score=predict(chosen,ref,q)
        evaluate(scheme,q,score,cal,y[ref],chosen,block)
        audit.append(dict(scheme=scheme,block=block,config=chosen,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum()),inner_AP=vals[chosen]))
        print('Completed',scheme,block,chosen,flush=True)
chosen,cal,vals=select(source)
score=predict(chosen,source,target_ids)
evaluate('RG3_to_CN7_source_selected',target_ids,score,cal,y[source],chosen,'CN7_all')
relative_choices=[i for i,c in enumerate(configs) if c['representation']=='relative_std4']
relative_chosen=max(relative_choices,key=lambda i:vals[i])
relative_cal=oof(relative_chosen,source)
evaluate('RG3_to_CN7_relative_source_selected',target_ids,predict(relative_chosen,source,target_ids),relative_cal,y[source],relative_chosen,'CN7_all')
pd.DataFrame([dict(config=i,**c,RG3_oof_AP=vals[i],selected_for_CN7=i==chosen) for i,c in enumerate(configs)]).to_csv(tables/'rg3_gas_tuning_candidates.csv',index=False,encoding='utf-8-sig')
pp=pd.DataFrame(predrows)
for (scheme,budget),part in pp.groupby(['scheme','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(part.actual,part.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,normal_calibration_budget=budget,shots=len(part),gas=int(part.actual.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(part.actual,part.score),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn),normal_FPR=fp/(fp+tn)))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_gas_tuning_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_gas_tuning_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_gas_tuning_fold_audit.csv',index=False,encoding='utf-8-sig')
lines=['# RG3 가스 RF 소규모 튜닝과 CN7 전이','',
'입력은 기존 이전10샷 표준편차4개, 또는 각각을 이전10샷 중앙값 절댓값(하한1)으로 나눈 상대 흔들림4개다. 중앙값은 현재값-dev10으로 복원한다. 현재 샷 이후 이력은 사용하지 않는다. 상대값은 제품 수준 차이를 줄이려는 후보이며 물리적 원인을 증명하지 않는다.',
'RF200개, seed42 유지. 깊이4/8, 최소잎3/8, 불량가중치 유무의 사전고정6조합 × 표현2개 =12후보. RG3 바깥 평가구간을 제외한 내부 구간검증 AP로 선택한다. 판정 기준은 내부 정상 점수의 상위1/3/5% 경계이며 모두 보고한다. 평가 오탐률을 보장하지 않는다.',
'CN7 적용 후보는 RG3 구간검증 AP만으로 선택한다. CN7 라벨/정상표본으로 모델이나 임계값을 조정하지 않는다. CN7의 이전센서이력은 입력 특성으로 사용한다. CN7 RH603(정상593/가스10); RG3 RH583(정상566/가스17), 미성형 제외.',
'CN7 가스10개가 한 관측구간에 몰려 있어 독립 구간으로 튜닝/평가를 동시에 나누기 어렵다. 이번에는 CN7 결과로 최적화하지 않는다. 양제품 원본센서이력을 확보할 수 있다는 조건이며 대회 테스트 가용성이 미확인이다.',
'RG3기간은 CN7보다 뒤이므로 제품전이이지 시간순 미래시험이 아니다. 동일자료 반복탐색 영향이 있으며 독립 최종성능이 아니다. 과거평가의 내부구간검증은 학습집합 안에서 양방향으로 수행한다.',
'','## CN7 적용 설정',str(configs[chosen]),'','## 결과','',
'| 평가 | 학습 정상 기준 | TP | FP | FN | TN | AP |','|---|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.scheme} | {r.normal_calibration_budget:.0%} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} |')
lines+=['','기존 결합 모델 구간제외 AP0.1686/TP4/FP6, 과거평가 AP0.0812/TP2/FP4와 비교한다. 이번은 결합 모델 전체 튜닝이 아니라 단독 흔들림 RF 및 상대값 후보 튜닝이다. 기준별 TP만으로 최고를 고르지 않는다.']
(root/'reports/rg3_gas_tuning_transfer.md').write_text('\n'.join(lines),encoding='utf-8')
print('SOURCE SELECTED',chosen,configs[chosen]);print(metrics.to_string(index=False))


                             scheme  normal_calibration_budget  shots  gas  TP  FP  FN  TN       AP  precision   recall  normal_FPR
         RG3_nested_leave_block_out                       0.01    583   17   4  22  13 544 0.100609   0.153846 0.235294    0.038869
         RG3_nested_leave_block_out                       0.03    583   17   5  50  12 516 0.100609   0.090909 0.294118    0.088339
         RG3_nested_leave_block_out                       0.05    583   17   6  78  11 488 0.100609   0.071429 0.352941    0.137809
               RG3_nested_past_only                       0.01    440   13   2  10  11 417 0.102699   0.166667 0.153846    0.023419
               RG3_nested_past_only                       0.03    440   13   4  27   9 400 0.102699   0.129032 0.307692    0.063232
               RG3_nested_past_only                       0.05    440   13   4  31   9 396 0.102699   0.114286 0.307692    0.072600
         RG3_to_CN7_source_selected                       0.01    603   10  